# Script for comparing different CNN models 

In [1]:
import mne
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score
import tensorflow as tf

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras import backend as K
import gc

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
mne.set_log_level("CRITICAL")

# Loading in Data

In [2]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 

trial_info = pd.read_csv('trial_info_duration_1205.csv')

# add duration information 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration_1', 'Duration_2']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)

features_all = features_all.rename(columns={
    'Duration_1': 'Duration_corr',
    'Duration_2': 'Duration_zygo'
})

features_all_store = features_all

x = np.isnan(features_all['Duration_zygo']) 
indices = np.where(x)[0]
features_all = features_all.drop(indices)

In [3]:
np.shape(features_all_store)
 

(7200, 18)

# Functions

In [4]:
# single head CNN model for number of contractions  
def CNN_model_contraction(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=3))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?
    model.add(MaxPooling1D(pool_size=1)) 

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [5]:
# single head CNN model for duration 
def CNN_model_duration(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=3))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(1, activation='linear'))


    return model  # Return the compiled model

In [6]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)


    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers


    shared = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    shared = Dropout(0.5)(shared)

    # Count-specific hidden layers
    count_branch = Dense(32, activation='relu')(shared)
    count_branch = Dropout(0.2)(count_branch)

    count_output = Dense(
        num_classes,
        activation='softmax',
        name='count_output'
    )(count_branch)

    duration_branch = Dense(32, activation='relu')(shared)
    duration_branch = Dropout(0.2)(duration_branch) # try increasing drop out for more regularization? (increase if overfitting)

    duration_output = Dense(
        1,
        activation='linear',
        name='duration_output'
    )(duration_branch)


    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

In [7]:
# single head CNN model for number of contractions  
def CNN_model_contraction_multichan(input_shape, num_classes,feature_num):
    global epoch_len
    
    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(16, kernel_size=3, activation='relu', input_shape=input_shape, kernel_regularizer=l2(0.0001))(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer
    x = Conv1D(32, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)



    x = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.7)(x)

    x = Flatten()(x)

    # two heads for each label 
    # output head for Zygo
    out_zygo = Dense(num_classes, activation='softmax', name="zygo_output", kernel_regularizer=l2(0.001))(x)

    # output head for Corr
    out_corr = Dense(num_classes, activation='softmax', name="corr_output", kernel_regularizer=l2(0.001))(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])
    
    return model 

In [8]:
def CNN_model_duration_multichan(input_shape, num_classes, feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(32, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(64, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)

    x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.4)(x)

    # output heads
    out_zygo = Dense(1, activation='relu', name="zygo_output")(x)
    out_corr = Dense(1, activation='relu', name="corr_output")(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])

    return model

In [9]:
def CNN_model_fourhead_multichan(input_shape, num_classes, feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # --- Shared CNN backbone ---
    x = Conv1D(32, kernel_size=3, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)
    shared = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    shared = Dropout(0.5)(shared)

    # zygo branch 
    zygo_branch = Dense(32, activation='relu')(shared)
    zygo_branch = Dropout(0.3)(zygo_branch)
    zygo_count_output    = Dense(num_classes, activation='softmax', name='zygo_count_output')(zygo_branch)
    zygo_duration_output = Dense(1, activation='linear', name='zygo_duration_output')(zygo_branch)

    # corr branch 
    corr_branch = Dense(32, activation='relu')(shared)
    corr_branch = Dropout(0.3)(corr_branch)
    corr_count_output    = Dense(num_classes, activation='softmax', name='corr_count_output')(corr_branch)
    corr_duration_output = Dense(1, activation='linear', name='corr_duration_output')(corr_branch)

    # --- Model ---
    model = Model(
        inputs=inputs,
        outputs=[
            zygo_count_output,
            zygo_duration_output,
            corr_count_output,
            corr_duration_output
        ]
    )

    return model

In [14]:
def run_kfold_training(
    model_func,
    X,
    y,
    input_shape,
    num_classes,
    feature_num,
    compile_kwargs,
    early_stop,
    fit_kwargs=None, # arguments fed to model.fit()
    n_splits=5,
    random_state=42,
    shuffle=True,
    verbose=1,
    type=1,
):
   # returns a dictionary with {"models", "histories", "cv_scores"}.     

    if fit_kwargs is None:
        fit_kwargs = {}
    fit_kwargs = fit_kwargs.copy()

 

    kf = KFold(n_splits=n_splits, random_state=random_state, shuffle=shuffle)
    models = []
    histories = []
    cv_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        print(f"Fold: {fold} {'='*65}")
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

   
        model = model_func(input_shape, num_classes, feature_num)
        model.compile(**compile_kwargs)

        if type ==2:
        
            history = model.fit(
            X_train,
        {"count_output": y_train[:, 0],
        "duration_output": y_train[:, 1],},
            validation_data=(X_val,  {
            "count_output": y_val[:, 0],
            "duration_output": y_val[:, 1],
        }
    ),
            callbacks=[early_stop],
            verbose=verbose,
            **fit_kwargs,
        )
            scores = model.evaluate(X_val, {
            "count_output": y_val[:, 0],
            "duration_output": y_val[:, 1],
        }, verbose=0, return_dict=True)
        elif type == 3:
            history = model.fit(
            X_train,
        {"zygo_output": y_train[:, 0],
        "corr_output": y_train[:, 1],},
            validation_data=(X_val,  {
            "zygo_output": y_val[:, 0],
            "corr_output": y_val[:, 1],
        }
    ),
            callbacks=[early_stop],
            verbose=verbose,
            **fit_kwargs,
        )
            scores = model.evaluate(X_val, {
            "zygo_output": y_val[:, 0],
            "corr_output": y_val[:, 1],
        }, verbose=0, return_dict=True)
        elif type == 4:
              history = model.fit(X_train, {
        'zygo_count_output': y_train[:, 0],
        'zygo_duration_output': y_train[:, 1],
        'corr_count_output': y_train[:, 2],
        'corr_duration_output': y_train[:, 3]
    }, epochs=10, validation_data=(
    X_val,
    {
        'zygo_count_output': y_val[:, 0],
        'zygo_duration_output': y_val[:, 1],
        'corr_count_output': y_val[:, 2],
        'corr_duration_output': y_val[:, 3]
    }
), callbacks=[early_stop])
              scores = model.evaluate(X_val, 
    {
        'zygo_count_output': y_val[:, 0],
        'zygo_duration_output': y_val[:, 1],
        'corr_count_output': y_val[:, 2],
        'corr_duration_output': y_val[:, 3]
    }, verbose=0, return_dict=True)
        else:
            history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            callbacks=[early_stop],
            verbose=verbose,
            **fit_kwargs,
        )
            
            scores = model.evaluate(X_val, y_val, verbose=0, return_dict=True)

        


        models.append(model)
        histories.append(history)
        cv_scores.append(scores)

    return {
        "models": models,
        "histories": histories,
        "cv_scores": cv_scores,
    }

Initializing Variables Needed

In [11]:
# global variables for training 
epoch_num = 5
patience = 3
early_stop = EarlyStopping(monitor='val_loss',  patience=patience, restore_best_weights=True)


metrics = ["f1", "accuracy", "mse"]
targets = ["zygo", "corr", "overall"]
splits = ["training", "testing"]
metric_tuples = [("model_name", "", "")]
metric_tuples += [
    (metric, target, split)
    for metric in metrics
    for target in targets
    for split in splits
]
metric_tuples += [("k-fold", "mean", ""), ("k-fold", "std", "")]
results_columns = pd.MultiIndex.from_tuples(
    metric_tuples,
    names=["metric", "target", "split"]
)
results_columns = pd.MultiIndex.from_tuples(metric_tuples)
results_df = pd.DataFrame(columns=results_columns)

# Single Channel Training 

## Single Head Count 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]
else:
    features_all_temp = features_all


muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_zygo = features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()

indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
# call function for training
results_single_head_contraction = run_kfold_training(
    model_func=CNN_model_contraction,
    X=X_train_full,
    y=y_train_full,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "metrics": ["accuracy"]
    },
    fit_kwargs={"epochs": 10},
    n_splits=5,
    early_stop=early_stop)

model_contraction = results_single_head_contraction["models"][-1]
model_history_contraction = model_contraction.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test), callbacks=[early_stop])

In [ ]:
# cross validation results 
cvScores_contraction = results_single_head_contraction["cv_scores"]
accuracies = [fold['accuracy'] for fold in cvScores_contraction]

avgScores = np.mean(accuracies)
stdScores = np.std(accuracies)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

# full training results (test data not seen during cross val)
y_pred_train = model_contraction.predict(X_train_full)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model_contraction.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   


# Calculate accuracy
accuracy_training = accuracy_score(y_train_full, y_pred_train)   
accuracy_test = accuracy_score(y_test, y_pred_test)  

# Calculate F1 score
f1_training = f1_score(y_train_full, y_pred_train, average='weighted')  
f1_test = f1_score(y_test, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Model scores---------------")
print("Training Accuracy :", accuracy_training)  
print("Test Accuracy :", accuracy_test)  
print("Training F1 Score :", f1_training)   
print("Test F1 Score :", f1_test) 

Count by Muscle Groups 

In [ ]:
zygo_ind_train = np.where(idx_train < 7140)
corr_ind_train = np.where(idx_train >= 7140)  
zygo_ind_test = np.where(idx_test < 7140)
corr_ind_test = np.where(idx_test >= 7140)  

In [ ]:
# remake splits (i think this is wrong)
X_train_full_corr = X[corr_ind_train]
X_train_full_zygo = X[zygo_ind_train]

X_test_corr = X[corr_ind_test]
X_test_zygo = X[zygo_ind_test]

y_train_full_corr = y[corr_ind_train]
y_train_full_zygo = y[zygo_ind_train]

y_test_corr = y[corr_ind_test]
y_test_zygo = y[zygo_ind_test]


# full training results (test data not seen during cross val)
y_pred_train_corr = model_contraction.predict(X_train_full_corr)  
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)   

y_pred_train_zygo = model_contraction.predict(X_train_full_zygo)  
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)   

# Predict on test data
y_pred_test_zygo = model_contraction.predict(X_test_zygo)   
y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)   

y_pred_test_corr= model_contraction.predict(X_test_corr)   
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)   

# Calculate accuracy
accuracy_training_corr = accuracy_score(y_train_full_corr, y_pred_train_corr)   
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)  

accuracy_training_zygo = accuracy_score(y_train_full_zygo, y_pred_train_zygo)   
accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)  

# Calculate F1 score
f1_training_corr = f1_score(y_train_full_corr, y_pred_train_corr, average='weighted')  
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')  

f1_training_zygo = f1_score(y_train_full_zygo, y_pred_train_zygo, average='weighted')  
f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')  


# Print accuracy and F1 score
print("Corru Scores------------------------------")  
print("Training Accuracy :", accuracy_training_corr) 
print("Test Accuracy :", accuracy_test_corr)  
print("Training F1 Score :", f1_training_corr)  
print("Test F1 Score :", f1_test_corr) 

print("Zygo Scores------------------------------")  
print("Training Accuracy :", accuracy_training_zygo)  
print("Test Accuracy :", accuracy_test_zygo)  
print("Training F1 Score :", f1_training_zygo)   
print("Test F1 Score :", f1_test_zygo)

In [ ]:
corr_mask = muscle_test == "Corr"
zygo_mask = muscle_test == "Zygo"

print("\nCorr Results-------------------")
#print("Accuracy Train:", accuracy_score(y_train_full[corr_mask], y_pred_train[corr_mask]))
print("Accuracy Test:", accuracy_score(y_test[corr_mask], y_pred_test[corr_mask]))
#print("F1 Train:", f1_score(y_train_full[corr_mask], y_pred_train[corr_mask]))
print("F1 Test:", f1_score(y_test[corr_mask], y_pred_test[corr_mask], average='weighted')  )

print("\nZygo Results-------------------")
#print("Accuracy Train:", accuracy_score(y_train_full[zygo_mask], y_pred_train[zygo_mask]))
print("Accuracy Test:", accuracy_score(y_test[zygo_mask], y_pred_test[zygo_mask])  )

#print("F1 Train:", f1_score(y_train_full[zygo_mask], y_pred_train[zygo_mask]))
print("F1 Test:", f1_score(y_test[zygo_mask], y_pred_test[zygo_mask], average='weighted')  )


Adding to dataframe

In [ ]:

new_row = {
    ("model_name", "", ""): "Single Head Contraction",

    # F1 scores
    ("f1", "zygo", "training"): f1_training_zygo,
    ("f1", "zygo", "testing"): f1_test_zygo,
    ("f1", "corr", "training"): f1_training_corr,
    ("f1", "corr", "testing"):f1_test_corr,
    ("f1", "overall", "training"):f1_training,
    ("f1", "overall", "testing"):f1_test,

    # Accuracy scores
    ("accuracy", "zygo", "training"):accuracy_training_zygo,
    ("accuracy", "zygo", "testing"): accuracy_test_zygo,
    ("accuracy", "corr", "training"): accuracy_training_corr,
    ("accuracy", "corr", "testing"):accuracy_test_corr,
    ("accuracy", "overall", "training"):(accuracy_training_zygo + accuracy_training_corr)/2,
    ("accuracy", "overall", "testing"): (accuracy_test_zygo + accuracy_test_corr)/2,

    # MAE scores
    ("mae", "zygo", "training"): np.nan,
    ("mae", "zygo", "testing"):np.nan,
    ("mae", "corr", "training"): np.nan,
    ("mae", "corr", "testing"): np.nan,
    ("mae", "overall", "training"): np.nan,
    ("mae", "overall", "testing"): np.nan,

    # K-fold
    ("k-fold", "mean", ""): avgScores,
    ("k-fold", "std", ""): stdScores,
}

results_df = pd.concat(
    [results_df, pd.DataFrame([new_row])],
    ignore_index=True
)

In [ ]:
# add the model results row to results_df
# then clear the model and free memory
del model_contraction,results_single_head_contraction
K.clear_session()
gc.collect()

## Single Head Duration

In [ ]:
# Define CNN model inputs
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all

muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = (np.concatenate((X_zygo, X_corr), axis=0))
y = (np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]]))     


indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
# call functionf or training 
results_single_head_dur = run_kfold_training(
    model_func=CNN_model_duration,
    X=X_train_full,
    y=y_train_full,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": "mae",
        "metrics": ["mae"]
    },
    fit_kwargs={"epochs": 10},
    n_splits=5,
    early_stop=early_stop)

model_duration = results_single_head_dur["models"][-1]
model_history_duration = model_duration.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test), callbacks=[early_stop])

In [ ]:
# cross validation results 
cvScores_duration = np.array(results_single_head_dur["cv_scores"])
mae = [fold['mae'] for fold in cvScores_duration]

avgScores = np.mean(mae)
stdScores = np.std(mae)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
 
baseline = np.mean(y_train_full)
mae_baseline = np.mean(np.abs(y_train_full - baseline))

# full training results (test data not seen during cross val)
y_pred_train_dur = model_duration.predict(X_train_full) 

# Predict on test data
y_pred_test_dur = model_duration.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full, y_pred_train_dur)   
mae_test_dur= mean_absolute_error(y_test, y_pred_test_dur)  
  
# Print accuracy and F1 score
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)

corr_mask = muscle_test == "Corr"
zygo_mask = muscle_test == "Zygo"

print("\nCorr Results-------------------")
print("MAE:", mean_absolute_error(y_test[corr_mask], y_pred_test_dur[corr_mask]))

print("\nZygo Results-------------------")
print("MAE:", mean_absolute_error(y_test[zygo_mask], y_pred_test_dur[zygo_mask]))
 

In [ ]:
# add to dataframe 
new_row = {
    ("model_name", "", ""): "Single Head Duration",

    # F1 scores
    ("f1", "zygo", "training"): np.nan,
    ("f1", "zygo", "testing"): np.nan,
    ("f1", "corr", "training"): np.nan,
    ("f1", "corr", "testing"):np.nan,
    ("f1", "overall", "training"):np.nan,
    ("f1", "overall", "testing"):np.nan,

    # Accuracy scores
    ("accuracy", "zygo", "training"):np.nan,
    ("accuracy", "zygo", "testing"): np.nan,
    ("accuracy", "corr", "training"): np.nan,
    ("accuracy", "corr", "testing"):np.nan,
    ("accuracy", "overall", "training"):np.nan,
    ("accuracy", "overall", "testing"): np.nan,

    # MAE scores
    ("mae", "zygo", "training"): np.nan,
    ("mae", "zygo", "testing"): mean_absolute_error(y_test[zygo_mask], y_pred_test_dur[zygo_mask]),
    ("mae", "corr", "training"): np.nan,
    ("mae", "corr", "testing"): mean_absolute_error(y_test[corr_mask], y_pred_test_dur[corr_mask]),
    ("mae", "overall", "training"): mae_training_dur,
    ("mae", "overall", "testing"): mae_test_dur,

    # K-fold
    ("k-fold", "mean", ""): avgScores,
    ("k-fold", "std", ""): stdScores,
}

results_df = pd.concat(
    [results_df, pd.DataFrame([new_row])],
    ignore_index=True
)

In [ ]:
del model_duration,results_single_head_dur
K.clear_session()
gc.collect()

## Two Head 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all

muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_durations = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       

y = np.column_stack((y_contractions, y_durations))

indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
# call function for training 
num_classes = len(np.unique(y_contractions))
train_epochs = 10


results_twohead = run_kfold_training(
    model_func=CNN_model_twohead,
    X=X_train_full,
    y=y_train_full,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": {
            "count_output": "sparse_categorical_crossentropy",
            "duration_output": "mae",
        },
        "metrics": {
            "count_output": ["accuracy"],
            "duration_output": ["mae"],
        },
    },
    fit_kwargs={"epochs": train_epochs},
    n_splits=5,
    early_stop=early_stop,
    type=2
)

cvScores = results_twohead["cv_scores"]
cvScores_contr = [score["count_output_accuracy"] * 100 for score in cvScores]
cvScores_dur = [score["duration_output_mae"] for score in cvScores]

model_twohead = results_twohead["models"][-1]
model_history_twohead = model_twohead.fit(
    X_train_full,
    {"count_output": y_train_full[:, 0], "duration_output": y_train_full[:, 1]},
    validation_data=(
        X_test,
        {"count_output": y_test[:, 0], "duration_output": y_test[:, 1]},
    ),
    epochs=train_epochs,
    callbacks=[early_stop],
)


In [ ]:
# comparison for two head model 
# cross validation results 
avgScores_dur = np.mean(cvScores_contr)
stdScores_dur = np.std(cvScores_contr)

avgScores_contr = np.mean(cvScores_dur)
stdScores_contr = np.std(cvScores_dur)

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {avgScores_dur}")

# full training results (test data not seen during cross val)
[y_pred_train_contraction, y_pred_train_dur] = model_twohead.predict(X_train_full)  

y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model_twohead.predict(X_test)  

y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   
 
# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full[:,0], y_pred_train_contraction)   
accuracy_test_contraction = accuracy_score(y_test[:,0], y_pred_test_contraction)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full[:,0], y_pred_train_contraction, average='weighted')  
f1_test_contraction = f1_score(y_test[:,0], y_pred_test_contraction, average='weighted')  

# MAE 
baseline = np.mean(y_train_full[:,1])
mae_baseline = np.mean(np.abs(y_train_full[:,1] - baseline))

#y_pred_train_dur = model_twohead.predict(X_train_full) 
#y_pred_test_dur = model_twohead.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full[:,1], y_pred_train_dur)   
mae_test_dur= mean_absolute_error(y_test[:,1], y_pred_test_dur)  
  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)   
print("----------------------") 
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)


In [ ]:
# add results to dataframe 
new_row = {
    ("model_name", "", ""): "Two Head",

    ("f1", "zygo", "training"): np.nan,
    ("f1", "zygo", "testing"): np.nan,
    ("f1", "corr", "training"): np.nan,
    ("f1", "corr", "testing"): np.nan,
    ("f1", "overall", "training"): f1_training_contraction,
    ("f1", "overall", "testing"): f1_test_contraction,

    ("accuracy", "zygo", "training"): np.nan,
    ("accuracy", "zygo", "testing"): np.nan,
    ("accuracy", "corr", "training"): np.nan,
    ("accuracy", "corr", "testing"): np.nan,
    ("accuracy", "overall", "training"): accuracy_training_contraction,
    ("accuracy", "overall", "testing"): accuracy_test_contraction,

    ("mse", "zygo", "training"): np.nan,
    ("mse", "zygo", "testing"): np.nan,
    ("mse", "corr", "training"): np.nan,
    ("mse", "corr", "testing"): np.nan,
    ("mse", "overall", "training"): mae_training_dur,
    ("mse", "overall", "testing"): mae_test_dur,

    ("k-fold", "mean", ""): [avgScores_dur, avgScores_contr],
    ("k-fold", "std", ""): [stdScores_dur,stdScores_contr],
}

results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

In [ ]:
del model_twohead,results_twohead
K.clear_session()
gc.collect()

# Two Channel 

## Count Multi Channel 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]
 

In [ ]:
two_channel_contraction = run_kfold_training(
    model_func=CNN_model_contraction_multichan,
    X=X_train_full,
    y=y_train_full,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": {
            "zygo_output": "sparse_categorical_crossentropy",
            "corr_output": "sparse_categorical_crossentropy",
        },
        "metrics": {
            "zygo_output": ["accuracy"],
            "corr_output": ["accuracy"],
        },
    },
    fit_kwargs={"epochs": 10},
    n_splits=5,
    early_stop=early_stop,
    type=3)

model_two_channel_contraction = two_channel_contraction["models"][-1]
cvScores = two_channel_contraction["cv_scores"]

model_history_twochannel_dur = model_two_channel_contraction.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

In [ ]:
cvScores

In [ ]:
accuracies_zygo = [fold['zygo_output_accuracy'] for fold in cvScores]
accuracies_corr = [fold['corr_output_accuracy'] for fold in cvScores]

avgScores_zygo = np.mean(accuracies_zygo,axis=0)
stdScores_zygo = np.std(accuracies_zygo)

avgScores_corr = np.mean(accuracies_corr,axis=0)
stdScores_corr = np.std(accuracies_corr)

avgScores_av =(avgScores_zygo + avgScores_corr) / 2
stdScores_av =(stdScores_zygo + stdScores_corr) / 2

print(f"Average KFold Cross Validation Score: {avgScores_av}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores_av}")
print("\n")

print(f"Average KFold Cross Validation Score Zygo: {avgScores_zygo}")
print(f"Standard Deviation KFold Cross Validation Score Zygo: {stdScores_zygo}")
print("\n")

print(f"Average KFold Cross Validation Score Corr: {avgScores_corr}")
print(f"Standard Deviation KFold Cross Validation Score Corr: {stdScores_corr}")
print("\n")

# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model_two_channel_contraction.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model_two_channel_contraction.predict(X_test)

# Convert probabilities to class labels
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)

# true labels for corr and zygo 
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 1]
y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 1]

# calculate accuracy for each muscle group 
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)

# calculate f1 score for each muscle group 
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')

print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n -------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)

# average score
print("\n -------- Average --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)  

In [ ]:

new_row = {
    ("model_name", "", ""): "Contraction Multichannel",

    # F1 scores
    ("f1", "zygo", "training"): f1_train_zygo,
    ("f1", "zygo", "testing"): f1_test_zygo,
    ("f1", "corr", "training"): f1_train_corr,
    ("f1", "corr", "testing"): f1_test_corr,
    ("f1", "overall", "training"): (f1_train_zygo + f1_train_corr) / 2,
    ("f1", "overall", "testing"): (f1_test_zygo + f1_test_corr) / 2,

    # Accuracy scores
    ("accuracy", "zygo", "training"):accuracy_train_zygo,
    ("accuracy", "zygo", "testing"): accuracy_test_zygo,
    ("accuracy", "corr", "training"): accuracy_train_corr,
    ("accuracy", "corr", "testing"):accuracy_test_corr,
    ("accuracy", "overall", "training"): (accuracy_train_zygo + accuracy_train_corr) / 2,
    ("accuracy", "overall", "testing"): (accuracy_test_zygo + accuracy_test_corr) / 2,

    # MAE scores
    ("mae", "zygo", "training"): np.nan,
    ("mae", "zygo", "testing"):np.nan,
    ("mae", "corr", "training"):np.nan,
    ("mae", "corr", "testing"): np.nan,
    ("mae", "overall", "training"): mae_training_dur,
    ("mae", "overall", "testing"): mae_test_dur,

    # K-fold
    ("k-fold", "mean", ""): [avgScores_zygo,avgScores_corr,avgScores_av],
    ("k-fold", "std", ""): [stdScores_zygo,stdScores_corr,stdScores_av],
}

results_df = pd.concat(
    [results_df, pd.DataFrame([new_row])],
    ignore_index=True
)

In [ ]:
del two_channel_contraction,model_two_channel_contraction
K.clear_session()
gc.collect()

mae_training_dur_zygo = mean_absolute_error(y_train_full[:, 0], y_pred_train_zygo)
mae_test_dur_zygo = mean_absolute_error(y_test[:, 0], y_pred_test_zygo)
mae_training_dur_corr = mean_absolute_error(y_train_full[:, 1], y_pred_train_corr)
mae_test_dur_corr = mean_absolute_error(y_test[:, 1], y_pred_test_corr)

new_row = {
    ("model_name", "", ""): "Two Channel Duration",

    ("f1", "zygo", "training"): np.nan,
    ("f1", "zygo", "testing"): np.nan,
    ("f1", "corr", "training"): np.nan,
    ("f1", "corr", "testing"): np.nan,
    ("f1", "overall", "training"): np.nan,
    ("f1", "overall", "testing"): np.nan,

    ("accuracy", "zygo", "training"): np.nan,
    ("accuracy", "zygo", "testing"): np.nan,
    ("accuracy", "corr", "training"): np.nan,
    ("accuracy", "corr", "testing"): np.nan,
    ("accuracy", "overall", "training"): np.nan,
    ("accuracy", "overall", "testing"): np.nan,

    ("mae", "zygo", "training"): mae_training_dur_zygo,
    ("mae", "zygo", "testing"): mae_test_dur_zygo,
    ("mae", "corr", "training"): mae_training_dur_corr,
    ("mae", "corr", "testing"): mae_test_dur_corr,
    ("mae", "overall", "training"): np.mean([mae_training_dur_zygo, mae_training_dur_corr]),
    ("mae", "overall", "testing"): np.mean([mae_test_dur_zygo, mae_test_dur_corr]),

    ("k-fold", "mean", ""): avgScores_av,
    ("k-fold", "std", ""): stdScores_av,
}

results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)Single Head Duration 

## Duration Multi channel

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [ ]:
two_channel_duration = run_kfold_training(
    model_func=CNN_model_duration_multichan,
    X=X_train_full,
    y=y_train_full,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": {
            "zygo_output": "mae",
            "corr_output": "mae",
        },
        "metrics": {
            "zygo_output": ["mae"],
            "corr_output": ["mae"],
        },
    },
    fit_kwargs={"epochs": 10},
    n_splits=5,
    early_stop=early_stop)


model_two_channel_duration = two_channel_duration["models"][-1]
cvScores = two_channel_duration["cv_scores"]
model_history_twochannel_dur = model_two_channel_duration.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

In [ ]:
cvScores

In [ ]:
# cross validation results 
# cross validation results 
accuracies_zygo = [fold['zygo_output_mae']/100 for fold in cvScores]
accuracies_corr = [fold['corr_output_mae']/100 for fold in cvScores]

avgScores_zygo = np.mean(accuracies_zygo,axis=0)
stdScores_zygo = np.std(accuracies_zygo)

avgScores_corr = np.mean(accuracies_corr,axis=0)
stdScores_corr = np.std(accuracies_corr)

avgScores_av =(avgScores_zygo + avgScores_corr) / 2
stdScores_av =(stdScores_zygo + stdScores_corr) / 2

print(f"Average KFold Cross Validation Score: {avgScores_av}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores_av}")
print("\n")

print(f"Average KFold Cross Validation Score Zygo: {avgScores_zygo}")
print(f"Standard Deviation KFold Cross Validation Score Zygo: {stdScores_zygo}")
print("\n")

print(f"Average KFold Cross Validation Score Corr: {avgScores_corr}")
print(f"Standard Deviation KFold Cross Validation Score Corr: {stdScores_corr}")
print("\n")

# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model_two_channel_duration.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model_two_channel_duration.predict(X_test)


baseline_zygo = np.mean(y_train_full[:,0])
mae_baseline_zygo = np.mean(np.abs(y_train_full[:,0] - baseline_zygo))

baseline_corr = np.mean(y_train_full[:,1])
mae_baseline_corr = np.mean(np.abs(y_train_full[:,1] - baseline_corr))
  
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,0], y_pred_train_zygo)   
mae_test_dur_zygo = mean_absolute_error(y_test[:,0], y_pred_test_zygo)  

mae_training_dur_corr = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo)   
mae_test_dur_corr= mean_absolute_error(y_test[:,1], y_pred_test_zygo)  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)
print("----------------------") 
print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)

In [ ]:

new_row = {
    ("model_name", "", ""): "Duration Multichannel",

    # F1 scores
    ("f1", "zygo", "training"): np.nan,
    ("f1", "zygo", "testing"): np.nan,
    ("f1", "corr", "training"): np.nan,
    ("f1", "corr", "testing"): np.nan,
    ("f1", "overall", "training"): np.nan,
    ("f1", "overall", "testing"): np.nan,

    # Accuracy scores
    ("accuracy", "zygo", "training"):np.nan,
    ("accuracy", "zygo", "testing"): np.nan,
    ("accuracy", "corr", "training"): np.nan,
    ("accuracy", "corr", "testing"):np.nan,
    ("accuracy", "overall", "training"):np.nan,
    ("accuracy", "overall", "testing"): np.nan,

    # MAE scores
    ("mae", "zygo", "baseline"): baseline_zygo,
    ("mae", "zygo", "training"): mae_training_dur_zygo,
    ("mae", "zygo", "testing"):mae_test_dur_zygo,
    ("mae", "corr", "baseline"): baseline_corr,
    ("mae", "corr", "training"):mae_training_dur_corr,
    ("mae", "corr", "testing"): mae_test_dur_corr,
    ("mae", "overall", "training"): (mae_training_dur_zygo+mae_training_dur_corr)/2,
    ("mae", "overall", "testing"): (mae_test_dur_zygo+mae_test_dur_corr)/2,

    # K-fold
    ("k-fold", "mean", ""): [avgScores_zygo,avgScores_corr,avgScores_av],
    ("k-fold", "std", ""): [stdScores_zygo,stdScores_corr,stdScores_av],
}

results_df = pd.concat(
    [results_df, pd.DataFrame([new_row])],
    ignore_index=True
)

In [ ]:
del model_two_channel_duration,two_channel_duration
K.clear_session()
gc.collect()

In [ ]:
results_df.to_pickle("model_results_df.pkl")

# Four Head 

In [12]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y_contr = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])


y_dur = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])


y = np.column_stack((y_contr, y_dur))
y = y[:, [0, 2, 1, 3]]

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [15]:
results_fourhead = run_kfold_training(
    model_func=CNN_model_fourhead_multichan,
    X=X_train_full,
    y=y_train_full,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": { 
        'zygo_count_output': 'sparse_categorical_crossentropy',
        'corr_count_output': 'sparse_categorical_crossentropy',
        'zygo_duration_output': 'mae',
        'corr_duration_output': 'mae'
        },
        "loss_weights":{
            'zygo_count_output':    1.0,
            'corr_count_output':    1.0,
            'zygo_duration_output': 0.01,
            'corr_duration_output': 0.01
        },
        "metrics": {
            'zygo_count_output': ['accuracy'],
        'corr_count_output': ['accuracy'],
        'zygo_duration_output': ['mae'],
        'corr_duration_output': ['mae']
        },
    },
    fit_kwargs={"epochs": 10},
    n_splits=5,
    early_stop=early_stop,
    type=4)

model_fourhead = results_fourhead["models"][-1]
cvScores = results_fourhead["cv_scores"]
model_history_fourhead = results_fourhead["histories"]

Fold: 1 =================================================================
Epoch 1/10
143/143 [==============================] - 16s 97ms/step - loss: 6.8368 - zygo_count_output_loss: 2.5826 - zygo_duration_output_loss: 44.3780 - corr_count_output_loss: 2.9241 - corr_duration_output_loss: 43.9143 - zygo_count_output_accuracy: 0.6612 - zygo_duration_output_mae: 44.3780 - corr_count_output_accuracy: 0.6032 - corr_duration_output_mae: 43.9143 - val_loss: 32.8880 - val_zygo_count_output_loss: 16.3945 - val_zygo_duration_output_loss: 4.7863 - val_corr_count_output_loss: 15.9935 - val_corr_duration_output_loss: 4.2205 - val_zygo_count_output_accuracy: 0.2170 - val_zygo_duration_output_mae: 4.7863 - val_corr_count_output_accuracy: 0.1767 - val_corr_duration_output_mae: 4.2205
Epoch 2/10
143/143 [==============================] - 12s 84ms/step - loss: 2.9866 - zygo_count_output_loss: 1.0377 - zygo_duration_output_loss: 31.9274 - corr_count_output_loss: 0.9816 - corr_duration_output_loss: 29.830

In [ ]:
model_fourhead = CNN_model_fourhead_multichan(input_shape, num_classes,feature_num)
epoch_num = 10 
early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
model_fourhead.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
    loss={
        'zygo_count_output': 'sparse_categorical_crossentropy',
        'corr_count_output': 'sparse_categorical_crossentropy',
        'zygo_duration_output': 'mae',
        'corr_duration_output': 'mae'
    },loss_weights={
            'zygo_count_output':    1.0,
            'corr_count_output':    1.0,
            'zygo_duration_output': 0.01,
            'corr_duration_output': 0.01
        },
    
    metrics={
        'zygo_count_output': ['accuracy'],
        'corr_count_output': ['accuracy'],
        'zygo_duration_output': ['mae'],
        'corr_duration_output': ['mae']
    }
)
model_history_fourhead = model_fourhead.fit(X_train_full, [
        y_train_full[:, 0],
        y_train_full[:, 1],
        y_train_full[:, 2],
        y_train_full[:, 3],
    ], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

Epoch 1/10


2026-05-18 23:41:53.610469: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


179/179 [==============================] - 29s 130ms/step - loss: 5.7284 - zygo_count_output_loss: 2.1259 - zygo_duration_output_loss: 39.6243 - corr_count_output_loss: 2.4493 - corr_duration_output_loss: 34.2934 - zygo_count_output_accuracy: 0.7234 - zygo_duration_output_mae: 39.6243 - corr_count_output_accuracy: 0.6607 - corr_duration_output_mae: 34.2934 - val_loss: 15.8537 - val_zygo_count_output_loss: 15.4151 - val_zygo_duration_output_loss: 4.3995 - val_corr_count_output_loss: 0.0000e+00 - val_corr_duration_output_loss: 0.0000e+00 - val_zygo_count_output_accuracy: 0.1225 - val_zygo_duration_output_mae: 4.3995 - val_corr_count_output_accuracy: 0.0000e+00 - val_corr_duration_output_mae: 0.0000e+00
Epoch 2/10
179/179 [==============================] - 17s 97ms/step - loss: 3.2547 - zygo_count_output_loss: 1.1499 - zygo_duration_output_loss: 33.0777 - corr_count_output_loss: 1.1011 - corr_duration_output_loss: 24.8511 - zygo_count_output_accuracy: 0.8190 - zygo_duration_output_mae: 33

In [ ]:
cvScores

[[1.4192832708358765,
  1.1533634662628174,
  0.47503864765167236,
  0.0,
  0.0,
  0.8460192680358887,
  0.47503864765167236,
  0.0,
  0.0],
 [1.3077647686004639,
  1.0097558498382568,
  0.41232189536094666,
  0.0,
  0.0,
  0.8635170459747314,
  0.41232189536094666,
  0.0,
  0.0],
 [1.0626299381256104,
  0.7905032634735107,
  0.6823802590370178,
  0.0,
  0.0,
  0.8327495455741882,
  0.6823802590370178,
  0.0,
  0.0],
 [0.980844259262085,
  0.6640825271606445,
  0.43332621455192566,
  0.0,
  0.0,
  0.8730297684669495,
  0.43332621455192566,
  0.0,
  0.0],
 [0.8881422877311707,
  0.6759617924690247,
  0.6545823216438293,
  0.0,
  0.0,
  0.8704028129577637,
  0.6545823216438293,
  0.0,
  0.0]]

In [ ]:
'''
# Cross-validation results
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
''' 
accuracies_zygo = [fold['zygo_output_accuracy'] for fold in cvScores]
accuracies_corr = [fold['corr_output_accuracy'] for fold in cvScores]

avgScores_zygo = np.mean(accuracies_zygo,axis=0)
stdScores_zygo = np.std(accuracies_zygo)

avgScores_corr = np.mean(accuracies_corr,axis=0)
stdScores_corr = np.std(accuracies_corr)

avgScores_av =(avgScores_zygo + avgScores_corr) / 2
stdScores_av =(stdScores_zygo + stdScores_corr) / 2

y_pred_train = model_fourhead.predict(X_train_full)
y_pred_test = model_fourhead.predict(X_test)

y_pred_train_zygo_count, y_pred_train_zygo_dur, y_pred_train_corr_count, y_pred_train_corr_dur = y_pred_train
y_pred_test_zygo_count, y_pred_test_zygo_dur, y_pred_test_corr_count, y_pred_test_corr_dur = y_pred_test


# baselines
baseline_zygo = np.mean(y_train_full[:,1])
baseline_corr = np.mean(y_train_full[:,3])

mae_baseline_zygo = np.mean(np.abs(y_train_full[:,1] - baseline_zygo))
mae_baseline_corr = np.mean(np.abs(y_train_full[:,3] - baseline_corr))

# MAE
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo_dur)
mae_test_dur_zygo = mean_absolute_error(y_test[:,1], y_pred_test_zygo_dur)

mae_training_dur_corr = mean_absolute_error(y_train_full[:,3], y_pred_train_corr_dur)
mae_test_dur_corr = mean_absolute_error(y_test[:,3], y_pred_test_corr_dur)

print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)

print("----------------------") 

print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)



y_pred_train_zygo_count = np.argmax(y_pred_train_zygo_count, axis=1)
y_pred_train_corr_count = np.argmax(y_pred_train_corr_count, axis=1)

y_pred_test_zygo_count = np.argmax(y_pred_test_zygo_count, axis=1)
y_pred_test_corr_count = np.argmax(y_pred_test_corr_count, axis=1)

# true labels
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 2]

y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 2]

# accuracy
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo_count)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr_count)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo_count)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr_count)

# f1
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo_count, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr_count, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo_count, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr_count, average='weighted')



print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n-------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)


print("\n-------- Average (Counts) --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)

print("\n-------- Average (Duration MAE) --------")
print("Training MAE:", (mae_training_dur_zygo + mae_training_dur_corr) / 2)
print("Test MAE:", (mae_test_dur_zygo + mae_test_dur_corr) / 2)

TypeError: list indices must be integers or slices, not str

In [ ]:

new_row = {
    ("model_name", "", ""): "Duration Multichannel",

    # F1 scores
    ("f1", "zygo", "training"): f1_train_zygo,
    ("f1", "zygo", "testing"): f1_test_zygo,
    ("f1", "corr", "training"): f1_train_corr,
    ("f1", "corr", "testing"): f1_test_corr,
    ("f1", "overall", "training"): (f1_train_zygo + f1_train_corr) / 2,
    ("f1", "overall", "testing"): (f1_test_zygo + f1_test_corr) / 2,

    # Accuracy scores
    ("accuracy", "zygo", "training"):accuracy_train_zygo,
    ("accuracy", "zygo", "testing"): accuracy_test_zygo,
    ("accuracy", "corr", "training"): accuracy_train_corr,
    ("accuracy", "corr", "testing"):accuracy_test_corr,
    ("accuracy", "overall", "training"): (accuracy_train_zygo + accuracy_train_corr) / 2,
    ("accuracy", "overall", "testing"): (accuracy_test_zygo + accuracy_test_corr) / 2,

    # MAE scores
    ("mae", "zygo", "baseline"): baseline_zygo,
    ("mae", "zygo", "training"): mae_training_dur_zygo,
    ("mae", "zygo", "testing"):mae_test_dur_zygo,
    ("mae", "corr", "baseline"): baseline_corr,
    ("mae", "corr", "training"):mae_training_dur_corr,
    ("mae", "corr", "testing"): mae_test_dur_corr,
    ("mae", "overall", "training"): (mae_training_dur_zygo+mae_training_dur_corr)/2,
    ("mae", "overall", "testing"): (mae_test_dur_zygo+mae_test_dur_corr)/2,

    # K-fold
    ("k-fold", "mean", ""): [cvScores_zygo,cvScores_corr,avgScores_av],
    ("k-fold", "std", ""): [stdScores_zygo,stdScores_corr,stdScores_av],
}

results_df = pd.concat(
    [results_df, pd.DataFrame([new_row])],
    ignore_index=True
)

NameError: name 'cvScores_zygo' is not defined

In [ ]:
del results_fourhead,model_fourhead
K.clear_session()
gc.collect()

In [ ]:
results_df.to_pickle("model_results_df_fourhead.pkl")

## Segmentation

def comparator(learner, instructor):
    if len(learner) != len(instructor):
        raise AssertionError("Layer count mismatch")
    for a, b in zip(learner, instructor):
        if tuple(a) != tuple(b):
            print(colored("Test failed", attrs=['bold']))
            raise AssertionError("Error in test")
    print(colored("All tests passed!", "green"))

def summary(model):
    result = []
    for layer in model.layers:
        output_shape = getattr(layer.output, 'shape', None)
        params = layer.count_params() if hasattr(layer, 'count_params') else 0
        result.append([layer.__class__.__name__, output_shape, params])
    return result
